# Dual-ring training

Simulate one ring pair at 1310 nm and train both CNNs to estimate its two ring frequencies.
The example uses the Site 3 layout and a second thermal coupling matrix for the transfer test.

In [ ]:
import copy
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

SEED = 17
TRAIN_STEPS = 4000
TRANSFER_STEPS = 1200
SAVE_DATASETS = False
OUT = Path("results/dual_ring")
OUT.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Simulate the spectra

The first two heater columns are the bus ring and the other ring.
The last four are the nearby aggressors. These are simulated heater powers in mW.

In [ ]:
# Site 3 heater centers in µm, in the order used by the dataset.
heater_xy = np.array([
    [698.287, 480.859],
    [685.711, 480.859],
    [485.595, 480.913],
    [498.405, 480.913],
    [885.595, 480.913],
    [898.405, 480.913],
])
distance = np.linalg.norm(heater_xy[:2, None] - heater_xy[None, :], axis=-1)
source_coupling = 2.4 * (0.95 * np.exp(-distance / 10) + 0.05 * np.exp(-distance / 300))
target_coupling = source_coupling * np.array([0.90, 1.10, 1.15, 1.15, 1.15, 1.15])

laser_ghz = 299792458 / 1310e-9 / 1e9
ghz_per_k = laser_ghz * 1.8e-4 / 4.2
ring_coupling = 20.0
intrinsic_loss = laser_ghz / (2 * 70_000)
bus_loss = laser_ghz / (2 * 35_000)
cold_ghz = np.array([120.0, 95.0])
freq_ghz = np.linspace(-250, 250, 1667, dtype=np.float32)


def supermodes(ring_freqs):
    half_detuning = (ring_freqs[..., 0] - ring_freqs[..., 1]) / 2
    half_split = np.sqrt((half_detuning + 0.5j * bus_loss) ** 2 + ring_coupling**2).real
    center = ring_freqs.mean(axis=-1)
    return np.stack([center - half_split, center + half_split], axis=-1)


def transmission_db(ring_freqs):
    bus = ring_freqs[..., 0, None]
    other = ring_freqs[..., 1, None]
    through = 1 - 2 * bus_loss / (
        1j * (freq_ghz - bus) + intrinsic_loss + bus_loss
        + ring_coupling**2 / (1j * (freq_ghz - other) + intrinsic_loss)
    )
    return 10 * np.log10(np.maximum(np.abs(through) ** 2, 1e-4))


def collect(coupling, count, seed):
    rng = np.random.default_rng(seed)
    powers = rng.uniform(0, 9.0, size=(count, 6))
    powers[:, 2:] *= rng.random((count, 4)) < 0.5
    ring_freqs = cold_ghz - ghz_per_k * (powers @ coupling.T)
    spectra = transmission_db(ring_freqs)
    spectra += rng.normal(0, 0.05, spectra.shape)
    modes = supermodes(ring_freqs)
    return {
        "spectra": spectra.astype(np.float32),
        "power_mw": powers.astype(np.float32),
        "ring_freqs": ring_freqs.astype(np.float32),
        "split": (modes[:, 1] - modes[:, 0]).astype(np.float32),
    }


source_train = collect(source_coupling, 12000, SEED)
source_val = collect(source_coupling, 1500, SEED + 1)
source_test = collect(source_coupling, 2500, SEED + 2)
target_cal = collect(target_coupling, 300, SEED + 3)
target_test = collect(target_coupling, 2000, SEED + 4)

if SAVE_DATASETS:
    datasets = {
        "source_train": source_train, "source_val": source_val, "source_test": source_test,
        "target_calibration": target_cal, "target_test": target_test,
    }
    for name, data in datasets.items():
        frame = pd.DataFrame(data["spectra"], columns=[f"S{i}" for i in range(len(freq_ghz))])
        for index, power in enumerate(data["power_mw"].T):
            frame[f"P{index} mW"] = power
        frame["f_bus GHz"], frame["f_other GHz"] = data["ring_freqs"].T
        frame["split GHz"] = data["split"]
        frame.to_csv(OUT / f"{name}.csv.gz", index=False)

## Train both networks

Normalize with the source training set. Errors are reported in GHz.

In [ ]:
class Niu(nn.Module):
    def __init__(self, points):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, 6), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, 6), nn.ReLU(), nn.MaxPool1d(2),
        )
        with torch.no_grad():
            size = self.features(torch.zeros(1, 1, points)).numel()
        self.hidden = nn.Sequential(nn.Flatten(), nn.Linear(size, 128), nn.ReLU())
        self.output = nn.Linear(128, 2)

    def forward(self, spectra):
        return self.output(self.hidden(self.features(spectra)))


class ConvNet(nn.Module):
    def __init__(self, points):
        super().__init__()
        layers = []
        for inputs, channels in [(1, 16), (16, 32), (32, 64)]:
            layers.extend([
                nn.Conv1d(inputs, channels, 15, padding=7),
                nn.BatchNorm1d(channels), nn.ReLU(), nn.MaxPool1d(2),
            ])
        self.features = nn.Sequential(*layers)
        self.features.eval()
        with torch.no_grad():
            size = self.features(torch.zeros(1, 1, points)).numel()
        self.hidden = nn.Sequential(nn.Flatten(), nn.Linear(size, 128), nn.ReLU())
        self.output = nn.Linear(128, 2)

    def forward(self, spectra):
        return self.output(self.hidden(self.features(spectra)))


spectrum_mean = float(source_train["spectra"].mean())
spectrum_std = float(source_train["spectra"].std())
freq_mean = source_train["ring_freqs"].mean(axis=0)
freq_std = source_train["ring_freqs"].std(axis=0)


def inputs(data):
    return torch.tensor((data["spectra"] - spectrum_mean) / spectrum_std, device=device).unsqueeze(1)


def labels(data):
    return torch.tensor((data["ring_freqs"] - freq_mean) / freq_std, device=device)


def fit(model, x, y, steps, lr, seed=SEED, head_only=False):
    model.train()
    if head_only:
        model.requires_grad_(False)
        model.output.requires_grad_(True)
        model.features.eval()

    params = [param for param in model.parameters() if param.requires_grad]
    optimizer = torch.optim.Adam(params, lr=lr)

    def lr_scale(step):
        return min(1, (step + 1) / 300) * (0.01 + 0.99 * (1 + math.cos(math.pi * step / steps)) / 2)

    schedule = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_scale)
    rng = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        idx = torch.randint(len(x), (min(32, len(x)),), generator=rng).to(device)
        optimizer.zero_grad()
        loss = nn.functional.mse_loss(model(x[idx]), y[idx])
        loss.backward()
        nn.utils.clip_grad_norm_(params, 1.0)
        optimizer.step()
        schedule.step()


@torch.no_grad()
def predict(model, x):
    model.eval()
    return torch.cat([model(batch).cpu() for batch in x.split(256)]).numpy()


def score(prediction, data):
    frequencies = prediction * freq_std + freq_mean
    error = frequencies - data["ring_freqs"]
    split_error = np.diff(supermodes(frequencies), axis=1).ravel() - data["split"]
    return {
        "ring_freq_rmse_GHz": float(np.sqrt(np.mean(error**2))),
        "ring_freq_mae_GHz": float(np.mean(np.abs(error))),
        "split_rmse_GHz": float(np.sqrt(np.mean(split_error**2))),
    }

In [ ]:
x_train, y_train = inputs(source_train), labels(source_train)
x_val, x_test = inputs(source_val), inputs(source_test)
x_cal, y_cal = inputs(target_cal), labels(target_cal)
x_target = inputs(target_test)

models = {}
validation_error = {}
test_scores = {}
for name, model_type in [("Niu", Niu), ("ConvNet", ConvNet)]:
    model = model_type(len(freq_ghz)).to(device)
    fit(model, x_train, y_train, TRAIN_STEPS, lr=1e-3)
    models[name] = model
    validation_error[name] = score(predict(model, x_val), source_val)["ring_freq_rmse_GHz"]
    test_scores[name] = score(predict(model, x_test), source_test)
    torch.save({
        "architecture": name,
        "weights": model.cpu().state_dict(),
        "freq_axis_GHz": freq_ghz.tolist(),
        "spectrum_mean": spectrum_mean,
        "spectrum_std": spectrum_std,
        "ring_freq_mean_GHz": freq_mean.tolist(),
        "ring_freq_std_GHz": freq_std.tolist(),
    }, OUT / f"{name}.pt")
    model.to(device)

best_name = min(validation_error, key=validation_error.get)
pd.DataFrame(test_scores).T

## Transfer to the second thermal condition

Compare the source model, an affine correction to its frequency predictions, a new output layer,
and a full fine-tune. All four use the same target test set.

In [ ]:
def affine(prediction, truth, strength):
    design = np.c_[prediction, np.ones(len(prediction))]
    prior = np.r_[np.eye(2), np.zeros((1, 2))]
    return np.linalg.solve(
        design.T @ design + strength * np.eye(3),
        design.T @ truth + strength * prior,
    )


rows = []
calibration_scores = []
y_cal_np = y_cal.cpu().numpy()
for name, model in models.items():
    cal_pred = predict(model, x_cal)
    target_pred = predict(model, x_target)
    weights = affine(cal_pred, y_cal_np, 1e-3)
    rows.append({"architecture": name, "method": "zero-shot", **score(target_pred, target_test)})
    rows.append({
        "architecture": name, "method": "affine adapter",
        **score(target_pred @ weights[:-1] + weights[-1], target_test),
    })

    head = copy.deepcopy(model)
    fit(head, x_cal, y_cal, TRANSFER_STEPS, 3e-4, seed=SEED + 1, head_only=True)
    rows.append({
        "architecture": name, "method": "output layer",
        **score(predict(head, x_target), target_test),
    })

    full = copy.deepcopy(model)
    fit(full, x_cal, y_cal, TRANSFER_STEPS, 1e-4, seed=SEED + 2)
    rows.append({
        "architecture": name, "method": "full fine-tune",
        **score(predict(full, x_target), target_test),
    })

    if name == best_name:
        for count in [4, 8, 16, 32, 64, 128, len(x_cal)]:
            for seed in range(3):
                idx = np.random.default_rng(100 + seed).choice(len(x_cal), count, replace=False)
                weights = affine(cal_pred[idx], y_cal_np[idx], 1e-2)
                calibration_scores.append({
                    "architecture": name, "n_calibration": count, "seed": seed,
                    **score(target_pred @ weights[:-1] + weights[-1], target_test),
                })

transfer = pd.DataFrame(rows)
calibration = pd.DataFrame(calibration_scores)
transfer.to_csv(OUT / "transfer_results.csv", index=False)
calibration.to_csv(OUT / "calibration_sweep.csv", index=False)
transfer

In [ ]:
calibration.groupby("n_calibration")["ring_freq_rmse_GHz"].mean().plot(marker="o", logx=True)
plt.xlabel("Target calibration spectra")
plt.ylabel("Ring-frequency RMSE (GHz)")
plt.show()

prediction = predict(models[best_name], x_target[:1]) * freq_std + freq_mean
plt.plot(freq_ghz, target_test["spectra"][0], label="target")
plt.plot(freq_ghz, transmission_db(prediction)[0], label="CNN estimate")
plt.xlabel("Frequency from laser (GHz)")
plt.ylabel("Transmission (dB)")
plt.legend()
plt.show()